# 网站摘要器（Website Summarizer）—— 本地 Ollama 版

## 练习目标（理念）

用 **Ollama** 的 OpenAI 兼容接口，对用户输入的 URL 做结构化网站摘要，并在笔记本里用 Markdown 展示。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions | `ollama.chat.completions.create(...)` |
| system / user messages | `system_prompt` + `user_prompt` + 抓取正文 |
| 本地模型 | `llama3.2`，`base_url=http://localhost:11434/v1` |
| 网页抓取 | `scraper.scrape_website` |

## 怎么跑

1. 确保本机 Ollama 已启动，并已 `ollama pull llama3.2`
2. 同目录有可用的 `scraper.py`
3. 从上到下运行；在 `input(...)` 处输入要摘要的网址


In [ ]:
# ========== 导入：工具箱 ==========

# 导入标准库 os：如需读环境变量可用（本笔记本主要走本地 Ollama）
import os
# 从 openai 导入 OpenAI 客户端：这里用来对接 Ollama 的 OpenAI 兼容 HTTP API
from openai import OpenAI
# 从本地 scraper 导入 scrape_website：给定 URL，返回网页正文文本
from scraper import scrape_website
# 从 IPython.display 导入 Markdown / display：在笔记本里渲染摘要
from IPython.display import Markdown, display


In [ ]:
# ========== 连接本地 Ollama（OpenAI 兼容 /v1）==========

# Ollama 默认提供的 OpenAI 兼容基址；路径 /v1 很重要
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# api_key='ollama'：本地服务通常不校验密钥，但 SDK 要求传一个非空字符串
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [ ]:
# ========== system prompt：规定「专家摘要器」的输出结构 ==========

# 发给模型的指令保持英文（翻译会改变行为）；要求 Markdown、分点、含受众等
system_prompt = """You are an expert website summarizer. Your task is to provide a structured, clear, and comprehensive summary of the website content provided.

Your summary must:
1. Provide a high-level overview of the website's purpose, main topic, or mission.
2. Extract the key features, main ideas, or services offered.
3. List the target audience or any notable call-to-actions, if applicable.
4. Organize the information logically using bullet points, headings, and clear formatting.
5. Answer strictly in Markdown format.
"""


In [ ]:
# ========== 主流程：输入 URL → 抓取 → 组 messages → 调本地模型 → 展示 ==========

# input：在笔记本里弹出提示，让你输入要摘要的网址（运行时交互）
website_url = input("Enter website URL you want to summarize: ")

# user 提示前缀：说明任务；可按需改措辞（字符串保持英文）
# 【注】You can update the user prompt accordinly
user_prompt = """
    Here are the contents of the website. Provide a short summary of this website.
    Mark the important information as well.
"""
# 抓取网页正文，拼进 user 消息
content = scrape_website(website_url)

# messages：system 定规则，user = 前缀 + 网页正文
messages = [
    { "role" : "system", "content" : system_prompt },
    { "role" : "user", "content" : user_prompt + content}
]

# 调用本地 llama3.2（model id 必须与 ollama list 中的名字一致）
response = ollama.chat.completions.create(model="llama3.2",messages=messages)
# 取出助手回复正文
summary = response.choices[0].message.content

# 用 Markdown 在笔记本中渲染摘要
display(Markdown(summary))
